# Entrenamiento y Evaluación de Modelos IDS
---
Este notebook contiene el motor de entrenamiento principal. Está diseñado para ser modular y aceptar diferentes arquitecturas (CNN, LSTM, Transformers, XGBoost, etc).

**Objetivos:**
1. Entrenar el modelo optimizando la función de pérdida `CrossEntropyLoss` o `Focal Loss`.
2. Monitorizar el `F1-Score Macro` para lidiar con el desbalanceo de clases.
3. Visualizar el rendimiento mediante curvas de aprendizaje y Matrices de Confusión.

#**Configuración del Entorno y Montaje de Drive**

In [ ]:
import sys
import os
import importlib

# 1. Montar Google Drive (Te pedirá permisos la primera vez)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("[-] No estamos en Google Colab. Ejecutando en local...")

# 2. Definir la ruta de tu proyecto
# ATENCIÓN: Confirma que esta es tu ruta exacta en Drive
PROJECT_PATH = '/content/drive/MyDrive/Codigo_Proyecto'

if IN_COLAB:
    # Añadir al path de Python para que reconozca la carpeta 'src'
    if PROJECT_PATH not in sys.path:
        sys.path.append(PROJECT_PATH)

    # Movernos físicamente a esa carpeta
    os.chdir(PROJECT_PATH)

print(f"[-] Directorio de trabajo actual: {os.getcwd()}")

# 3. Forzar RECARGA de módulos (El truco antimagia)
# Esto garantiza que si modificas un modelo en VS Code y guardas (Ctrl+S),
# Colab coja la versión nueva instantáneamente sin tener que reiniciar el entorno.
import src.config
import src.data_pipeline_extended

importlib.reload(src.config)
importlib.reload(src.data_pipeline_extended)

print("Entorno conectado y módulos sincronizados con VS Code.")

#**Importaciones y Configuración Gráfica**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import numpy as np
import time
import os
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from IPython.display import display, HTML

from src.config import Config
from src.data_pipeline_extended import get_dataloaders, ATTACK_MAPPING

# Crear carpeta para modelos
os.makedirs(Config.MODELS_PATH, exist_ok=True)

# Configuración visual de gráficas
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 12, 'figure.dpi': 120})

print(f"Entorno listo. Usando acelerador:  {Config.DEVICE}")

#**CARGA DE DATOS Y Cálculo de efectividad de pesos del dataset**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
import torch
from src.data_pipeline_extended import get_dataloaders, ATTACK_MAPPING, Config

print(" 1. Generando DataLoaders (Esto puede tardar un par de minutos por el RUS + SMOTE Tomek)...")
train_loader, val_loader, test_loader = get_dataloaders()

# Invertimos el diccionario para poder imprimir los nombres luego
idx_to_class = {v: k for k, v in ATTACK_MAPPING.items()}
class_names = [idx_to_class[i] for i in range(len(idx_to_class))]
print(f" [-] Clases detectadas para evaluación: {class_names}")

#SISTEMA DE PESOS HÍBRIDO
print("\n 2. Calculando pesos de clase para penalizar errores en ataques raros reales...")

pesos_manuales = [
    1.0,  # 0: Normal Traffic
    1.0,  # 1: DoS (Bajado para evitar confusión con Web)
    1.0,  # 2: DDoS
    1.2,  # 3: Port Scanning
    1.5,  # 4: Brute Force
    2.5,  # 5: Web Attacks (Alta prioridad)
    2.5   # 6: Bots
]

# Convertir a Tensor
pesos_tensor = torch.tensor(pesos_manuales, dtype=torch.float).to(Config.DEVICE)

print("\nRESUMEN DE PESOS (Multiplicador de castigo al modelo si falla):")
for i, peso in enumerate(pesos_manuales):
    nombre_clase = idx_to_class.get(i, "Desconocida")
    print(f"   -> Clase {i} ({nombre_clase}): {peso:.2f}x")

#**FOCAL LOSS**
- Experimental

In [ ]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss: La función para centrarse en clases difíciles.
    Fórmula: FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    """
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        """
        Args:
            alpha (Tensor, optional): Pesos de clase (para desbalanceo).
            gamma (float): Factor de enfoque. 2.0 es el estándar.
                           Cuanto más alto, más ignora los ejemplos fáciles.
            reduction (str): 'mean' (promedio) o 'sum' (suma).
        """
        super(FocalLoss, self).__init__()
        self.gamma = gamma
        self.reduction = reduction
        self.alpha = alpha

    def forward(self, inputs, targets):
        # 1. Calcular Cross Entropy básica (sin reducir todavía)
        # inputs: Logits del modelo (sin softmax previa)
        # targets: Etiquetas reales
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')

        # 2. Calcular la probabilidad de acierto (pt)
        # pt es la probabilidad que el modelo asignó a la clase correcta
        pt = torch.exp(-ce_loss)

        # 3. Calcular el factor focal: (1 - pt)^gamma
        # Si pt es alto (el modelo está seguro), el factor es cercano a 0 (pérdida se anula).
        # Si pt es bajo (el modelo falla), el factor es cercano a 1 (pérdida se mantiene).
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss

        # 4. Aplicar reducción final
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

#**El Motor de Entrenamiento**

In [ ]:
def train_model(model, model_name, train_loader, val_loader, epochs=100, lr=1e-3, patience=10, resume=True, class_weights=None):
    model = model.to(Config.DEVICE)

    # selección del criterio por si se vuelve arrogante el modelo con falsos positivos y falla Validation Loss
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=pesos_tensor, label_smoothing=0.1)
        print(" Entrenando con Pesos Suavizados + Label Smoothing (Optimizado para Adversarial)")
    else:
        criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
        print("Entrenando sin pesos")

    # 1. Optimizador AdamW (Mejor regularización que Adam)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # 2. Scheduler: Reduce el LR si el F1-Score no mejora en 5 épocas
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.1, patience=5)

    # 3. Activación Focal Loss si está activado
    #criterion = FocalLoss(alpha=pesos_tensor, gamma=2.0)

    best_model_path = os.path.join(Config.MODELS_PATH, f"{model_name}_best.pth")
    checkpoint_path = os.path.join(Config.MODELS_PATH, f"{model_name}_checkpoint.pth")

    start_epoch = 0
    best_val_f1 = 0.0
    epochs_no_improve = 0 # Contador para Early Stopping
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}

    # --- LÓGICA DE REANUDACIÓN ---
    if resume and os.path.exists(checkpoint_path):
        print(f"Checkpoint encontrado. Reanudando {model_name}...")
        checkpoint = torch.load(checkpoint_path, map_location=Config.DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_val_f1 = checkpoint['best_val_f1']
        history = checkpoint['history']
        epochs_no_improve = checkpoint.get('epochs_no_improve', 0)
        print(f"Reanudado en Época {start_epoch} (Mejor F1: {best_val_f1:.4f})")

    if start_epoch >= epochs:
        print("El modelo ya completó todas las épocas solicitadas.")
        return model, history, best_model_path

    print(f"\nIniciando entrenamiento: {model_name}")
    epoch_pbar = tqdm(range(start_epoch, epochs), desc="Progreso Global", initial=start_epoch, total=epochs)

    for epoch in epoch_pbar:
        # --- TRAIN ---
        model.train()
        train_loss = 0.0
        train_batches = tqdm(train_loader, desc=f"Época {epoch+1}/{epochs} [Train]", leave=False)

        for X_batch, y_batch in train_batches:
            X_batch, y_batch = X_batch.to(Config.DEVICE), y_batch.to(Config.DEVICE)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()

            # Gradient Clipping (Evita que los gradientes exploten, muy útil con LSTMs)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)

        train_loss /= len(train_loader.dataset)
        history['train_loss'].append(train_loss)

        # --- VALIDATION ---
        model.eval()
        val_loss, all_preds, all_labels = 0.0, [], []

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(Config.DEVICE), y_batch.to(Config.DEVICE)
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * X_batch.size(0)

                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())

        val_loss /= len(val_loader.dataset)
        val_f1 = f1_score(all_labels, all_preds, average='macro')

        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)

        # 3. Paso del Scheduler basado en la métrica de validación
        scheduler.step(val_f1)

        # --- LÓGICA DE EARLY STOPPING Y GUARDADO ---
        saved_msg = ""
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            epochs_no_improve = 0 # Reseteamos la paciencia
            torch.save(model.state_dict(), best_model_path)
            saved_msg = " # (Mejor modelo!) #"
        else:
            epochs_no_improve += 1
            saved_msg = f" (Sin mejora: {epochs_no_improve}/{patience})"

        # Checkpoint de seguridad
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_val_f1': best_val_f1,
            'history': history,
            'epochs_no_improve': epochs_no_improve
        }, checkpoint_path)

        current_lr = optimizer.param_groups[0]['lr']
        epoch_pbar.set_postfix({'T_Loss': f"{train_loss:.4f}", 'V_F1': f"{val_f1:.4f}", 'LR': f"{current_lr:.1e}"})
        print(f"Época {epoch+1:02d} | T.Loss: {train_loss:.4f} | V.Loss: {val_loss:.4f} | V.F1: {val_f1:.4f} | LR: {current_lr:.1e}{saved_msg}")

        # 4. Trigger de Early Stopping
        if epochs_no_improve >= patience:
            print(f"\n EARLY STOPPING TRIGGERED! El modelo dejó de mejorar durante {patience} épocas consecutivas.")
            break

    print(f"\nFin del entrenamiento. Mejor F1-Score (Validación): {best_val_f1:.4f}")
    return model, history, best_model_path

#Entrenamiento MultiScale-CNN con ArcFace

In [ ]:
def train_model_arcface(model, model_name, train_loader, val_loader, epochs=50, lr=1e-3, patience=10, resume=True):
    """
    Función de entrenamiento ESPECIAL para ArcFace.
    Diferencia clave: Pasa (inputs, labels) al modelo en el forward.
    """
    model = model.to(Config.DEVICE)

    # ArcFace suele funcionar mejor con SGD + Momentum o AdamW
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    # Scheduler: Reduce LR si se estanca
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)

    # Usamos CrossEntropyLoss estándar.
    # La magia de ArcFace ocurre DENTRO del modelo, que ya devuelve los logits modificados.
    criterion = nn.CrossEntropyLoss()

    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = 0.0
    patience_counter = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'lr': []}

    # --- LÓGICA DE RESUME ---
    save_path = f"{model_name}_best.pth"

    if resume and os.path.exists(save_path):
        print(f"🔄 Reanudando entrenamiento desde: {save_path}")
        try:
            # Cargar pesos
            model.load_state_dict(torch.load(save_path))
            best_model_wts = copy.deepcopy(model.state_dict())

            # Nota: Recuperar el mejor F1 exacto es complicado sin guardar metadatos,
            # así que asumimos un valor alto seguro o recalculamos.
            # Para PoC, basta con cargar los pesos y seguir.
            print("   -> Pesos cargados correctamente.")
        except Exception as e:
            print(f"⚠️ Error al cargar checkpoint: {e}. Empezando de cero.")

    print(f"🚀 Iniciando entrenamiento de ARCFACE: {model_name}")

    for epoch in range(epochs):
        model.train() # Modo entrenamiento
        running_loss = 0.0

        for inputs, labels in train_loader:
            inputs = inputs.to(Config.DEVICE)
            labels = labels.to(Config.DEVICE) # Necesitamos etiquetas en GPU

            optimizer.zero_grad()

            # --- CAMBIO CRÍTICO PARA ARCFACE ---
            # Pasamos inputs Y labels. El modelo usa labels para aplicar el margen angular.
            outputs = model(inputs, labels)

            loss = criterion(outputs, labels)
            loss.backward()

            #Gradient Clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_loader.dataset)
        history['train_loss'].append(epoch_loss)

        # --- VALIDACIÓN ---
        model.eval() # Modo evaluación
        val_loss = 0.0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(Config.DEVICE)
                labels = labels.to(Config.DEVICE)

                # --- CAMBIO CRÍTICO EN VALIDACIÓN ---
                # Pasamos labels=None. ArcFace actúa como clasificador normal (coseno).
                outputs = model(inputs, labels=None)

                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)

                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        epoch_val_loss = val_loss / len(val_loader.dataset)
        history['val_loss'].append(epoch_val_loss)

        # Métricas
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        history['val_f1'].append(val_f1)
        history['lr'].append(optimizer.param_groups[0]['lr'])

        scheduler.step(val_f1)

        print(f"Ep {epoch+1:02d} | T.Loss: {epoch_loss:.4f} | V.Loss: {epoch_val_loss:.4f} | V.F1: {val_f1:.4f}")

        # Guardar mejor modelo
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
            # Guardado rápido
            torch.save(model.state_dict(), f"{model_name}_best.pth")
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("🛑 Early Stopping activado.")
            break

    # Cargar mejores pesos
    model.load_state_dict(best_model_wts)
    return model, history, f"{model_name}_best.pth"

#**Funciones de Visualización**

In [ ]:
def plot_training_history(history, model_name):
    """Genera gráficas de las curvas de aprendizaje."""
    epochs = range(1, len(history['train_loss']) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Curvas de Aprendizaje: {model_name}", fontsize=16, fontweight='bold')

    # Gráfica de Pérdida (Loss)
    ax1.plot(epochs, history['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(epochs, history['val_loss'], 'r--', label='Validation Loss', linewidth=2)
    ax1.set_title('Evolución del Error (Loss)')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Loss')
    ax1.legend()

    # Gráfica de F1-Score
    ax2.plot(epochs, history['val_f1'], 'g-', label='Validation F1-Score Macro', linewidth=2)
    ax2.set_title('Evolución del Rendimiento (F1-Score)')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('F1-Score')
    ax2.legend()

    plt.tight_layout()
    plt.show()

def evaluate_and_plot(model, test_loader, model_path, model_name, class_names):
    """
    Evalúa el modelo en Test y dibuja la Matriz de Confusión.
    Args:
        class_names (list): Lista de strings con los nombres de los ataques en orden.
    """
    print("\n" + "="*50)
    print("EVALUACIÓN FINAL EN CONJUNTO DE TEST (DATOS NUEVOS)")
    print("="*50)

    # Cargar los mejores pesos
    model.load_state_dict(torch.load(model_path))
    model = model.to(Config.DEVICE)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(Config.DEVICE), y_batch.to(Config.DEVICE)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    # 1. Mostrar Reporte de Clasificación como DataFrame bonito
    report_dict = classification_report(all_labels, all_preds, target_names=class_names, output_dict=True)
    report_df = pd.DataFrame(report_dict).transpose()
    display(HTML(report_df.to_html(classes='table table-striped table-hover', float_format=lambda x: f'{x:.4f}')))

    # 2. Matriz de Confusión Visual
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Matriz de Confusión - {model_name}", fontsize=14, fontweight='bold')
    plt.ylabel('Etiqueta Real')
    plt.xlabel('Predicción del Modelo')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

#**Ejecución del Modelo**
- Two-Branch CNN

In [ ]:
from src.models.two_branch import TwoBranchCNN

model_name = "Two_Branch_CNN"
modelo = TwoBranchCNN(num_features=52, num_classes=7)

# entrenar
trained_model, history, saved_path = train_model(
    model=modelo,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=60,                    # limite alto, luego corta el Early Stop
    lr=1e-4,                      # learning rate incial, luego reducirá dependiendo metricas
    patience=12,
    #class_weights=pesos_tensor,
    class_weights=None,
    resume=True                  # esto empieza de cero el entrenamiento
)

# gráficas
plot_training_history(history, model_name)

# evaluaciones y matriz confusión
evaluate_and_plot(trained_model, test_loader, saved_path, model_name, class_names)

#Tabular ResNet

In [ ]:
from src.models.tabular_resnet import TabularResNet

model_name = "Tabular_ResNet"
modelo = TabularResNet(
    num_features=52,
    num_classes=7,
    hidden_dim=256,
    num_blocks=3,
    dropout_rate=0.2
)

# entrenar (modificar épocas)
trained_model, history, saved_path = train_model(
    model=modelo,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=60,                    # limite alto, luego corta el Early Stop
    lr=1e-4,                      # learning rate incial, luego reducirá dependiendo metricas
    patience=10,                  # épocas para que scheduler reduzca LR
    #class_weights=pesos_tensor,
    class_weights=None,
    resume=False                 # esto empieza de cero el entrenamiento
)

# gráficas
plot_training_history(history, model_name)

# evaluaciones y matriz confusión
evaluate_and_plot(trained_model, test_loader, saved_path, model_name, class_names)

#Multi-Scale CNN con Squeeze-and-Excitation (SE-Block)

In [ ]:
from src.models.multi_scale_cnn import MultiScaleSE_CNN

model_name = "MultiScale_SE_CNN_PoC"
modelo = MultiScaleSE_CNN(
    num_features=52,
    num_classes=7
).to(Config.DEVICE)

# Entrenamiento estándar (recomiendo lr=1e-3 al principio porque tiene BatchNorm)
trained_model, history, saved_path = train_model(
    model=modelo,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=60,
    lr=1e-3,              # primero 1e-3, si oscila bajarlo a 1e-4
    patience=15,
    class_weights=None,   # dejar que el se-block filtre el ruido
    resume=True
)

# gráficas
plot_training_history(history, model_name)

# evaluaciones y matriz confusión
evaluate_and_plot(trained_model, test_loader, saved_path, model_name, class_names)

In [ ]:
# gráficas
plot_training_history(history, model_name)

# evaluaciones y matriz confusión
evaluate_and_plot(trained_model, test_loader, saved_path, model_name, class_names)

#Advanced Multi-Scale CNN + ArcFace

In [ ]:
from src.models.advanced_multiscale_cnn_arcface import AdvancedNIDSModel

model_name = "Advanced_Multi-Scale_CNN_ArcFace"
modelo = AdvancedNIDSModel(
    num_features=52,
    num_classes=7
).to(Config.DEVICE)

# Entrenamiento estándar (recomiendo lr=1e-3 al principio porque tiene BatchNorm)
trained_model, history, saved_path = train_model_arcface(
    model=modelo,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=60,                    # limite alto, luego corta el Early Stop
    lr=1e-4,                      # learning rate incial, luego reducirá dependiendo metricas
    patience=12,                  # épocas para que scheduler reduzca LR
    resume=True                  # esto empieza de cero el entrenamiento
)

# gráficas
plot_training_history(history, model_name)

# evaluaciones y matriz confusión
evaluate_and_plot(trained_model, test_loader, saved_path, model_name, class_names)

##ME ASEGURO QUE GUARDO LOS MODELOS
- Guarda el estado del último modelo entrenado. Cuidado con entrenar otro modelo y ponerle el nombre de otro sin queerer

In [ ]:
# 1. Guardar pesos del modelo (ya lo hace la función, pero por seguridad)
torch.save(modelo.state_dict(), "ArcFace_Titan_Final_80percent.pth")

# 2. Guardar también la arquitectura entera (opcional, útil para cargar rápido mañana)
torch.save(modelo, "ArcFace_Titan_Full_Model.pth")

#EXTRAER TENSORES DE PROCESAMIENTO DE DATOS Y CONVERTIRLOS A NUMPY
- Aquí si guardamos los tensores, en la carpeta data/processed

In [ ]:
import numpy as np
import torch
import os

# Ya lo compruebo al principio del notebook, pero por si acaso
OUTPUT_DIR = '/content/drive/MyDrive/Codigo_Proyecto/data/processed'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Extrayeendo datos del DataLoader...")

# 2. Extraer todos los datos del val_loader (o test_loader)
all_inputs = []
all_labels = []

# Iteramos sobre tu val_loader para sacar los tensores
for inputs, labels in val_loader: # Cambia val_loader por test_loader si es el que usas
    all_inputs.append(inputs.cpu().detach().numpy())
    all_labels.append(labels.cpu().detach().numpy())

# 3. Concatenar todo en un solo array gigante de NumPy
x_val_np = np.concatenate(all_inputs, axis=0)
y_val_np = np.concatenate(all_labels, axis=0)

print(f"Datos extraídos. Shape de X: {x_val_np.shape} | Shape de Y: {y_val_np.shape}")

# 4. Guardar los arrays en Google Drive
ruta_x = os.path.join(OUTPUT_DIR, 'x_val_poc.npy')
ruta_y = os.path.join(OUTPUT_DIR, 'y_val_poc.npy')

np.save(ruta_x, x_val_np)
np.save(ruta_y, y_val_np)

print(f"¡Guardado exitoso!")
print(f"X guardado en: {ruta_x}")
print(f"Y guardado en: {ruta_y}")

# (Opcional) Guardar un "mini-batch" de 500 muestras para la demo de Streamlit
np.save(os.path.join(OUTPUT_DIR, 'demo_samples.npy'), x_val_np[:500])
np.save(os.path.join(OUTPUT_DIR, 'demo_labels.npy'), y_val_np[:500])